# High Tight Flag Breakout on SPY
## Strategy Brief
The High Tight Flag Breakout strategy seeks to identify and capitalize on strong upward trends in the SPY ETF. The strategy looks for a significant price increase followed by a tight consolidation, indicating a potential continuation of the uptrend. When the price breaks out of this consolidation pattern, a buy signal is triggered. The goal is to capture gains from the continuation of the bullish trend. This strategy is designed to outperform buy-and-hold by capturing explosive moves while minimizing exposure during consolidations.
## References
- (No external references)

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## PHASE 1 - Trading Context
In this phase, we define the parameters and constants used throughout the notebook. These include the stock ticker, date range, and any specific thresholds or conditions relevant to the High Tight Flag Breakout strategy.

In [ ]:
# Configuration
TICKER = 'SPY'
START_DATE = '2010-01-01'
END_DATE = '2023-10-31'
BREAKOUT_THRESHOLD = 0.05  # 5% breakout threshold
CONSOLIDATION_PERIOD = 20  # 20 days consolidation period
MIN_FLAGPOLE_GAIN = 0.25  # Minimum 25% gain for flagpole

## PHASE 2 - Data Exploration
We will download historical price data for SPY using yfinance, calculate relevant indicators, and visualize the data to understand potential breakout patterns.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Download SPY data
data = yf.download(TICKER, start=START_DATE, end=END_DATE)

# Calculate daily returns
data['Returns'] = data['Adj Close'].pct_change()

# Calculate rolling max to identify potential flagpoles
data['RollingMax'] = data['Adj Close'].rolling(window=CONSOLIDATION_PERIOD).max()

# Plot price and rolling max
plt.figure(figsize=(14, 7))
plt.plot(data['Adj Close'], label='SPY Price')
plt.plot(data['RollingMax'], label='Rolling Max', linestyle='--')
plt.title('SPY Price and Rolling Max')
plt.legend()
plt.show()

## PHASE 3 - Strategy Engineering
In this phase, we define the logic for identifying high tight flag breakout patterns and determine the entry and exit signals based on these patterns.

In [ ]:
# Identify flagpole: price increases by at least MIN_FLAGPOLE_GAIN within CONSOLIDATION_PERIOD
data['Flagpole'] = (data['Adj Close'] / data['Adj Close'].shift(CONSOLIDATION_PERIOD) - 1) > MIN_FLAGPOLE_GAIN

# Identify breakout: price exceeds previous rolling max by BREAKOUT_THRESHOLD
data['Breakout'] = (data['Adj Close'] > data['RollingMax'] * (1 + BREAKOUT_THRESHOLD))

# Combine flagpole and breakout conditions
data['Signal'] = data['Flagpole'] & data['Breakout']

# Define positions: 1 for buy, 0 for hold
positions = data['Signal'].astype(int)

## PHASE 4 - Coding & Backtesting
We will simulate the strategy by shifting the positions to avoid look-ahead bias, calculate daily returns, and visualize the equity curve.

In [ ]:
# Shift positions to avoid look-ahead bias
positions = positions.shift(1).fillna(0)

# Calculate strategy returns
data['StrategyReturns'] = positions * data['Returns']

data['EquityCurve'] = (1 + data['StrategyReturns']).cumprod()

# Plot equity curve
plt.figure(figsize=(14, 7))
plt.plot(data['EquityCurve'], label='Strategy Equity Curve')
plt.title('Strategy Equity Curve')
plt.legend()
plt.show()

## PHASE 5 - Performance Evaluation
Evaluate the performance of the strategy using key metrics such as CAGR, Sharpe ratio, Sortino ratio, Calmar ratio, and maximum drawdown. Compare these metrics to a simple buy-and-hold strategy.

In [ ]:
def calculate_performance_metrics(equity_curve):
    # Calculate CAGR
    total_return = equity_curve.iloc[-1] / equity_curve.iloc[0] - 1
    num_years = (equity_curve.index[-1] - equity_curve.index[0]).days / 365.25
    cagr = (1 + total_return) ** (1 / num_years) - 1
    
    # Calculate Sharpe ratio
    sharpe_ratio = equity_curve.pct_change().mean() / equity_curve.pct_change().std() * np.sqrt(252)
    
    # Calculate Sortino ratio
    downside_std = equity_curve.pct_change()[equity_curve.pct_change() < 0].std()
    sortino_ratio = equity_curve.pct_change().mean() / downside_std * np.sqrt(252)
    
    # Calculate Calmar ratio
    max_drawdown = (equity_curve.cummax() - equity_curve).max() / equity_curve.cummax().max()
    calmar_ratio = cagr / max_drawdown
    
    return cagr, sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown

# Calculate performance metrics for strategy
strategy_metrics = calculate_performance_metrics(data['EquityCurve'])

# Calculate performance metrics for buy-and-hold
buy_and_hold_equity_curve = (1 + data['Returns']).cumprod()
buy_and_hold_metrics = calculate_performance_metrics(buy_and_hold_equity_curve)

# Display comparison table
performance_df = pd.DataFrame({'Strategy': strategy_metrics, 'Buy and Hold': buy_and_hold_metrics},
                              index=['CAGR', 'Sharpe Ratio', 'Sortino Ratio', 'Calmar Ratio', 'Max Drawdown'])
print(performance_df)

## PHASE 6 - Deploy & Monitor
Create a function to download recent data, compute today's signal, and determine the current position.

In [ ]:
def get_current_signal():
    # Download last 60 days of data
data = yf.download(TICKER, period='60d')

    # Calculate rolling max
    data['RollingMax'] = data['Adj Close'].rolling(window=CONSOLIDATION_PERIOD).max()

    # Identify flagpole and breakout
    flagpole = (data['Adj Close'].iloc[-1] / data['Adj Close'].iloc[-CONSOLIDATION_PERIOD] - 1) > MIN_FLAGPOLE_GAIN
    breakout = data['Adj Close'].iloc[-1] > data['RollingMax'].iloc[-1] * (1 + BREAKOUT_THRESHOLD)

    # Determine signal
    signal = flagpole and breakout

    # Print current position
    position = 1 if signal else 0
    print(f"Current position for {TICKER}: {'Buy' if position == 1 else 'Hold'}")

get_current_signal()